# Notebook 25 — LiveMath-Judge exploratory diagnostic (NOT a scoring run)

> **The gate in notebook 24 FAILED.** On item 273 the judge accepted `3/4`
> against the page's `2/4` — it recalculated, which is exactly what its own
> criterion 3 and our fidelity clause both forbid.
>
> **Nothing in this notebook may be used for headline accuracy.** It exists to
> characterise *how* the judge fails, not to score with it. No cell computes an
> accuracy figure and the summary writer refuses to emit one.

Notebook 24 keeps its gate and stays the scoring path. This is deliberately a
separate file so an exploratory run can never be mistaken for, or reuse, the
gated one.

**Sample**: 40 items — 20 `has_error=1`, 20 clean, items 55 and 273 forced in,
spread round-robin across seven answer shapes (MCQ, derivative, set, system,
multi-value, text conclusion, numeric/algebra) and preferring items that carry
a determinate human label. Seeded, never hand-picked.

**Both prompts are run** — the model's native card template and the
fidelity-amended one — so the effect of the amendment is measured rather than
assumed. 80 judgments, a few minutes.

`strict_v1` and `strict_v2` are untouched.

In [ ]:
# Auth + code access. GPU: LiveMath-Judge is a 3B Qwen2.5 fine-tune.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"
OUT_DIR = f"{PROJECT_DIR}/audit"
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.audit_diagnostics
import pilot.canonicalize
import pilot.judge
import pilot.strict_v2

assert pilot.canonicalize.latex_parser_available(), "SymPy LaTeX parser broken"
import torch
assert torch.cuda.is_available(), "no GPU: enable a GPU runtime"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  pilot from "
      f"{os.path.dirname(pilot.judge.__file__)}")

In [ ]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

run = pd.read_csv(f"{RESULTS_DIR}/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv")
assert len(run) == 300

tokenizer = AutoTokenizer.from_pretrained(pilot.judge.MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    pilot.judge.MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
    token=HF_TOKEN).eval()


def make_backend(max_new_tokens=pilot.judge.MAX_NEW_TOKENS):
    """str -> str. Two corrections to the published snippet: return_dict=True
    (it returns a TENSOR and then subscripts it) and an explicit
    max_new_tokens (its default of 20 truncates before the boxed verdict).
    Only NEW tokens are decoded, so the prompt's own \boxed{yes} can never be
    read as the answer."""
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def backend(prompt: str) -> str:
        msgs = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            msgs, return_tensors="pt", return_dict=True,
            add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=pad_id)
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True)

    return backend


backend = make_backend()
print("backend ready")

In [ ]:
# The stratified 40. Seeded; 55 and 273 forced in and COUNTED toward the
# has_error stratum so the totals stay exactly 20/20.
v2 = pilot.strict_v2.rescore_v2(run, progress=True)

audits = pilot.audit_diagnostics.load_audit_sets("repo/reference/audit")
dedup = {}
for name in pilot.audit_diagnostics.SET_PRECEDENCE:
    for i, r in audits[name].iterrows():
        dedup.setdefault(i, r["truth"])
human = pd.Series(dedup)
human = human[human != "indeterminate"]

sample = pilot.judge.diagnostic_sample(run, v2, human)
assert len(sample) == 40
assert int(sample["has_error"].sum()) == 20
assert set(pilot.judge.GATE_ITEMS) <= set(sample.index)
print(f"{len(sample)} items | has_error {int(sample['has_error'].sum())} / "
      f"clean {int((~sample['has_error']).sum())} | "
      f"human-labelled {int((sample['human_label'] != '').sum())}")
print(sample["answer_type"].value_counts().to_string())
print(f"\nitems: {sample.index.tolist()}")

In [ ]:
# Both prompts on the 40 = 80 judgments. NO GATE HERE, by design: the judge
# is already known unsafe and the point is to watch it fail.
import ast

from tqdm.auto import tqdm

rows = []
for i in tqdm(sample.index, desc="native+fidelity"):
    raws = ast.literal_eval(run.loc[i, "all_transcription_samples_raw"])
    gt, q = run.loc[i, "pert_a"], run.loc[i, "orig_q"]
    nat = pilot.judge.judge_item(backend, raws, gt, q, fidelity=False)
    fid = pilot.judge.judge_item(backend, raws, gt, q, fidelity=True)
    rows.append({
        "item": int(i),
        "has_error": bool(sample.loc[i, "has_error"]),
        "answer_type": sample.loc[i, "answer_type"],
        "human_label": sample.loc[i, "human_label"],
        "forced": bool(sample.loc[i, "forced"]),
        "ground_truth_answer": fid["ground_truth_answer"],
        "model_answer": fid["model_answer"],
        "verdict_native": nat["verdict"],
        "verdict_fidelity": fid["verdict"],
        "label_native": nat["livemath_label"],
        "label_fidelity": fid["livemath_label"],
        "raw_native": nat["livemath_raw_output"],
        "raw_fidelity": fid["livemath_raw_output"],
        "solving_native": pilot.judge.looks_like_solving(nat["livemath_raw_output"]),
        "solving_fidelity": pilot.judge.looks_like_solving(fid["livemath_raw_output"]),
    })

both = pd.DataFrame(rows).set_index("item")
CSV_PATH = f"{OUT_DIR}/livemath_judge_diagnostic40_20260812.csv"
both.reset_index().to_csv(CSV_PATH, index=False)
print(f"\nper-item -> {CSV_PATH}")
print(f"prompts disagree on {int((both.verdict_native != both.verdict_fidelity).sum())} of 40")
print(f"judge appears to solve on "
      f"{int((both.solving_native | both.solving_fidelity).sum())} of 40")

In [ ]:
# Summary. Reports NO accuracy -- the gate failed and this is exploratory.
MD_PATH = f"{OUT_DIR}/livemath_judge_diagnostic40_summary_20260812.md"
text = pilot.judge.diagnostic_summary_md(MD_PATH, both, sample)
print(text[:4000])
print("\n" + "=" * 70)
print("Download both from Drive > uncertainty-math-vlm > audit/ into the repo")
print("at reference/audit/ under the same names.")
print("\nThe two probes, for the record:")
for i in pilot.judge.GATE_ITEMS:
    r = both.loc[i]
    print(f"  item {i}: native={r['verdict_native']}  fidelity={r['verdict_fidelity']}"
          f"  solving={bool(r['solving_fidelity'])}")
    print(f"     {str(r['raw_fidelity'])[:300]}")